In [21]:
import numpy as np
from collections import Counter
from ArcMemory import ObjectState, PixelSet, ArcState
from helpers import determine_new_obj_props, mask_to_pixels
import scipy.ndimage as ndi
from ArcDSL import _update_object_state

In [2]:
def fill_enclosed_area(
    obj: ObjectState,
    out_colour: int,
) -> ObjectState:
    """
    Fill any enclosed area of the object with the specified colour.
    Return a new ObjectState with the filled area (this will be a different object state than the input object as could be a different colour and area)
    """
    # Create a binary mask of the object
    binary_mask = obj.mask != 0
    filled_mask = ndi.binary_fill_holes(binary_mask)
    # Create a new array with the filled areas set to the specified colour
    filled_array = np.where(filled_mask, out_colour, obj.mask)
    new_pixels = determine_new_obj_props(filled_array, obj.centroid)
    return _update_object_state(obj, PixelSet(new_pixels))

In [3]:
train = {
            "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ],
            "output": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                [0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ],
}

In [4]:
arc_state = ArcState.from_array(np.array(train["input"]))
obj = arc_state.objects[1]


In [5]:
obj

ObjectState(label_id=1, colour=2, grid_size=(10, 15), bounding_box=(2, 3, 7, 6), centroid=(4.0, 3.625), area=8, cell_positions=frozenset({(4, 4), (3, 4), (5, 4), (2, 3), (4, 5), (3, 3), (5, 3), (6, 3)}), hu_moments=(-0.6054662398447882, -1.7927132059713304, -1.9914381551770042, -3.3746558659085792, 6.057653435124202, 4.271011762566332, -0.0), mutation_types=frozenset(), mutation_vectors=(), priority=0)

In [6]:
def pointed_direction(self) -> tuple[int, int]:
    """
    Direction a triangle / pyramid points: toward its narrow (apex) end.
    Returns (0, 0) if the shape isn't a pyramid along either axis.
    """
    rows = [int(r) for r, _ in self.cell_positions]
    cols = [int(c) for _, c in self.cell_positions]
    row_count = Counter(rows)
    col_count = Counter(cols)

    top, bottom = row_count[min(rows)], row_count[max(rows)]
    left, right = col_count[min(cols)], col_count[max(cols)]

    if top < bottom:
        return (-1, 0)   # apex at top -> points up
    if bottom < top:
        return (1, 0)    # apex at bottom -> points down
    if left < right:
        return (0, -1)   # apex at left -> points left
    if right < left:
        return (0, 1)    # apex at right -> points right
    return (0, 0)

def add_new_obj_line(obj: ObjectState, out_colour: int):
    """
    Add a new object that is a straight line extending from original object.
    """
    rows, cols = obj.grid_size
    dr, dc = pointed_direction(obj)
    if (dr, dc) == (0, 0):
        return obj
    if dr == 0:
        edge_col = (
            max(c for r, c in obj.cell_positions)
            if dc > 0
            else min(c for r, c in obj.cell_positions)
        )
        end_row = max(r for r, c in obj.cell_positions if c == edge_col)
        start = (end_row, edge_col)

    else:
        edge_row = (
            max(r for r, c in obj.cell_positions)
            if dr > 0
            else min(r for r, c in obj.cell_positions)
        )
        end_col = max(c for r, c in obj.cell_positions if r == edge_row)
        start = (edge_row, end_col)
    print(f"Adding new line from {start} in direction ({dr}, {dc})")
    new_cell_positions = set()
    r, c = start
    for i in range(1, max(rows, cols)):
        nr, nc = r + i * dr, c + i * dc
        if 0 <= nr < rows and 0 <= nc < cols:
            new_cell_positions.add((nr, nc))
        else:
            break
    if not new_cell_positions:
        return obj

    bbox, centroid, hu = determine_new_obj_props(new_cell_positions)
    return ObjectState(
        label_id=np.random.randint(1, 1000),
        colour=out_colour,
        grid_size=obj.grid_size,
        bounding_box=bbox,
        centroid=centroid,
        area=len(new_cell_positions),
        cell_positions=PixelSet(new_cell_positions),
        hu_moments=hu,
    )



In [7]:
pointed_direction(obj)

(0, 1)

In [8]:
add_new_obj_line(obj, out_colour=2)

Adding new line from (4, 5) in direction (0, 1)


ObjectState(label_id=161, colour=2, grid_size=(10, 15), bounding_box=(4, 6, 5, 15), centroid=(4.0, 10.0), area=9, cell_positions=frozenset({(4, 10), (4, 13), (4, 9), (4, 6), (4, 12), (4, 8), (4, 14), (4, 11), (4, 7)}), hu_moments=array([0.74074074, 0.54869684, 0.        , 0.        , 0.        ,
       0.        , 0.        ]), mutation_types=frozenset(), mutation_vectors=(), priority=0)

In [9]:
def remove_empty_outer_rows_and_columns(
    input_array: np.ndarray, background_colour: int
) -> np.ndarray:
    """
    Remove empty outer rows and columns from the input array. Defined as only background colour
    The removed rows and columns are only those that are on the outer edges of the array, not any empty rows or columns that may be in the middle of the array.
    """
    rows, cols = input_array.shape
    top, bottom = 0, rows - 1
    left, right = 0, cols - 1
    new_array = input_array.copy()
    # Remove empty outer rows
    while top <= bottom and np.all(new_array[top, :] == background_colour):
        new_array = np.delete(new_array, top, axis=0)
        bottom -= 1
    while bottom >= top and np.all(new_array[bottom, :] == background_colour):
        new_array = np.delete(new_array, bottom, axis=0)
        bottom -= 1
    # Remove empty outer columns
    while left <= right and np.all(new_array[:, left] == background_colour):
        new_array = np.delete(new_array, left, axis=1)
        right -= 1
    while right >= left and np.all(new_array[:, right] == background_colour):
        new_array = np.delete(new_array, right, axis=1)
        right -= 1
    return new_array

In [10]:
train = {           "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 3, 0, 0, 0, 0, 0, 3, 0],
                [0, 0, 0, 2, 2, 0, 0, 0, 0],
                [0, 0, 0, 2, 2, 0, 2, 0, 0],
                [0, 0, 2, 0, 0, 2, 0, 0, 0],
                [0, 3, 0, 0, 0, 0, 0, 3, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0],
            ]}

In [11]:
remove_empty_outer_rows_and_columns(np.array(train["input"]), background_colour=0)

array([[3, 0, 0, 0, 0, 0, 3],
       [0, 0, 2, 2, 0, 0, 0],
       [0, 0, 2, 2, 0, 2, 0],
       [0, 2, 0, 0, 2, 0, 0],
       [3, 0, 0, 0, 0, 0, 3]])

In [30]:
def count_colours(
    colour_count: Counter,
) -> np.ndarray:
    """
    Use the colour counter to return a grid with the counts
    Return as a horizontal/vertical line (dependent on output shape) of pixels with a new row for each colour and the count as the length of the line
    """
    if not colour_count:
        return None
    if len(colour_count) == 1:
        return None
    n_colours = len(colour_count)
    max_count = max(colour_count.values())
    colour_count = Counter(
        dict(sorted(colour_count.items(), key=lambda x: x[1], reverse=False))
    )
    output_array = np.zeros((n_colours, max_count), dtype=int)
    for i, (colour, count) in enumerate(colour_count.items()):
        output_array[i, :count] = colour
    return output_array

In [32]:
count_colours(Counter({1: 3, 2: 5, 3: 2}))

array([[3, 3, 0, 0, 0],
       [1, 1, 1, 0, 0],
       [2, 2, 2, 2, 2]])

In [26]:
def fill_enclosed_area(
    obj: ObjectState,
    out_colour: int,
) -> ObjectState:
    """
    Fill any enclosed area of the object with the specified colour.
    Return a new ObjectState with the filled area (this will be a different object state than the input object as could be a different colour and area)
    """
    # Create a binary mask of the object
    binary_mask = obj.mask != 0
    filled_mask = ndi.binary_fill_holes(binary_mask)
    # Create a new array with the filled areas set to the specified colour
    filled_array = np.where(filled_mask, out_colour, obj.mask)
    new_pixels = mask_to_pixels(filled_array)
    if out_colour == obj.colour:
        combined_pixels = obj.cell_positions.union(new_pixels)
        return _update_object_state(obj, combined_pixels)
    else:
        return [ObjectState(
            label_id=obj.label_id + np.random.randint(1, 1000),
            colour=out_colour,
            grid_size=obj.grid_size,
            bounding_box=obj.bounding_box,
            centroid=obj.centroid,
            area=len(new_pixels),
            cell_positions=PixelSet(new_pixels),
        ),
        obj,
        ]

In [29]:
train ={            "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ]}

obj = ArcState.from_array(np.array(train["input"])).objects[0]
new_obj = fill_enclosed_area(obj, out_colour=2)[0]
# Recreate grid with new cell positions
new_obj_cell_positions = new_obj.cell_positions
grid = np.zeros(obj.grid_size, dtype=int)
for r, c in new_obj_cell_positions:
    grid[r, c] = new_obj.colour
for r, c in obj.cell_positions:
    grid[r, c] = obj.colour  # Add original object colour

# Pretty print the grid
print("\n".join(" ".join(str(cell) for cell in row) for row in grid))

0 0 0 0 0 0 0 0 0 0 8 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 8 0 0 0 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 8 0 0 0 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 8 0 0 0 0 0 8 0 0 0 0 0 0
0 0 0 8 8 8 8 8 8 8 8 8 8 0 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 8 2 2 2 8 0 0 0 0 0 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 8 0 0 0 0 0 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 8 0 0 0 0 0 8 0 0 0 8 0 0
0 0 0 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 0
0 0 0 0 0 0 8 2 2 2 8 2 2 2 2 2 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 8 2 2 2 2 2 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 8 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 2 2 2 2 2 2 2 2 2 8 0 0 0 0 0 0
8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 0 8 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 0 8 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 0 8 2 2 8 0 0 0 0 0 0
0 0 0 0 0 0 8 0 0 0 0 0 8 8 8 8 8 8 8 8 8 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 8 0 0 8 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 8 0 0 8 

In [109]:
def _unfill_object(
    obj: ObjectState,
) -> ObjectState:
    """
    Unfill the object, only keeping the border of the object and setting the inside to background colour
    """
    cell_positions = obj.cell_positions
    new_cell_positions = set()
    # Get border cells only
    for cell in cell_positions:
        x, y = cell
        neighbors = [
            (x - 1, y),
            (x + 1, y),
            (x, y - 1),
            (x, y + 1),
        ]
        if any(neighbor not in cell_positions for neighbor in neighbors):
            new_cell_positions.add(cell)
    new_cell_positions = PixelSet(new_cell_positions)
    return _update_object_state(obj, new_cell_positions)


def unfill_all_objects(state: ArcState) -> ArcState:
    """
    Unfill all objects in the state, only keeping the borders of the objects and setting the inside to background colour
    """
    new_objects = []
    for obj in state.objects:
        new_obj = _unfill_object(obj)
        new_objects.append(new_obj)
    return ArcState(
        grid_state=state.grid_state,
        objects=tuple(new_objects),
    )


In [115]:
train = {             "input": [
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 8, 8, 8, 8, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
            ],}

arc_state = ArcState.from_array(np.array(train["input"]))

new_state = unfill_all_objects(arc_state)

# Recreate grid with new cell positions
grid = np.zeros(arc_state.grid_state.dimensions, dtype=int)
for obj in new_state.objects:
    for r, c in obj.cell_positions:
        grid[r, c] = obj.colour

grid

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 8, 8, 8, 8, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 7, 7, 0],
       [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0, 7, 0, 7, 0],
       [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0, 7, 0, 7, 0],
       [0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 7, 0, 7, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0,

In [118]:
def join_single_pixels_to_large_colour_object(
    state: ArcState
):
    """
    Move and join single pixels onto a large object of the same colour if available
    """
    grid = state.to_array()
    objects = state.objects
    new_grid = grid.copy()
    single_pixel_objects = [obj for obj in objects if obj.area == 1]
    large_colour_objects = [obj for obj in objects if obj.area > 1]

    if not single_pixel_objects or not large_colour_objects:
        return state

    for single_pixel in single_pixel_objects:
        colour = single_pixel.colour
        # Find large object of same colour
        large_object = [obj for obj in large_colour_objects if obj.colour == colour]
        if not large_object:
            continue
        else:
            large_object = large_object[0]
        single_pixel_pos = list(single_pixel.cell_positions)[0]
        # Look for shared row or column with large object
        large_object_rows = [pos[0] for pos in large_object.cell_positions]
        large_object_cols = [pos[1] for pos in large_object.cell_positions]
        if single_pixel_pos[0] in large_object_rows:
            # Slide along row to join (not merge)
            new_grid[single_pixel_pos[0], single_pixel_pos[1]] = 0
            target_col = (min(large_object_cols) - 1) if single_pixel_pos[1] < min(large_object_cols) else max(large_object_cols) + 1
            new_grid[single_pixel_pos[0], target_col] = colour
        elif single_pixel_pos[1] in large_object_cols:
            # Slide along column to join (not merge)
            new_grid[single_pixel_pos[0], single_pixel_pos[1]] = 0
            target_row = (min(large_object_rows) - 1) if single_pixel_pos[0] < min(large_object_rows) else max(large_object_rows) + 1
            new_grid[target_row, single_pixel_pos[1]] = colour
    return ArcState.from_array(new_grid)

In [121]:
train = {            "input": [
                [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0],
                [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
                [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3],
                [2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
                [2, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 3],
                [2, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
                [2, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 3],
                [2, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 2, 0, 0, 3],
                [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
                [0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0],
            ],}

arc_state = ArcState.from_array(np.array(train["input"]))
new_grid = join_single_pixels_to_large_colour_object(arc_state)
new_grid.to_array()

array([[0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0],
       [2, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 3],
       [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3],
       [2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
       [2, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 3],
       [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3],
       [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
       [2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3],
       [2, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 3],
       [0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0]])